In [1]:
# Cell 1: install compatible versions into the CURRENT notebook kernel
%pip install -q numpy==1.26.4 tensorflow==2.16.1 shap==0.45.1 opencv-python==4.9.0.80 scikit-learn pandas matplotlib jupyter ipykernel
print("Installation finished. Please RESTART the kernel before running the next cell.")

Note: you may need to restart the kernel to use updated packages.
Installation finished. Please RESTART the kernel before running the next cell.


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\Asus\\anaconda3\\envs\\xai_clean\\Lib\\site-packages\\tensorflow\\compiler\\mlir\\quantization\\tensorflow\\python\\pywrap_function_lib.pyd'
Consider using the `--user` option or check the permissions.



In [1]:
# Cell 2: verify environment after restarting kernel
import sys
print("Python executable:", sys.executable)

import numpy as np
print("NumPy version:", np.__version__)

import tensorflow as tf
print("TensorFlow version:", tf.__version__)

import shap
print("SHAP version:", shap.__version__)

Python executable: c:\Users\Asus\anaconda3\envs\xai_clean\python.exe
NumPy version: 1.26.4


ImportError: DLL load failed while importing _pywrap_tf2: The specified module could not be found.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import shap

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

## Set dataset paths

Expected folder structure:

```text
dataset/
    train/
        NORMAL/
        PNEUMONIA/
    val/
        NORMAL/
        PNEUMONIA/
    test/
        NORMAL/
        PNEUMONIA/
```

In [ ]:
DATASET_DIR = "dataset"

train_dir = os.path.join(DATASET_DIR, "train")
val_dir = os.path.join(DATASET_DIR, "val")
test_dir = os.path.join(DATASET_DIR, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42

print("Train folder exists:", os.path.exists(train_dir))
print("Val folder exists:", os.path.exists(val_dir))
print("Test folder exists:", os.path.exists(test_dir))

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=True,
    seed=SEED
)

val_data = test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class indices:", train_data.class_indices)

In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")]
)

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=3
)

In [ ]:
test_loss, test_acc, test_precision, test_recall = model.evaluate(test_data)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test precision: {test_precision:.4f}")
print(f"Test recall: {test_recall:.4f}")

pred_probs = model.predict(test_data)
pred_labels = (pred_probs > 0.5).astype(int).flatten()
true_labels = test_data.classes

print(classification_report(true_labels, pred_labels, target_names=list(test_data.class_indices.keys())))

cm = confusion_matrix(true_labels, pred_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(test_data.class_indices.keys()))
disp.plot()
plt.show()

## Grad-CAM

In [ ]:
def get_img_array(img_path, size):
    img = load_img(img_path, target_size=size)
    arr = img_to_array(img)
    arr = np.expand_dims(arr, axis=0)
    return preprocess_input(arr)

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    superimposed_img = cv2.addWeighted(img, 1 - alpha, heatmap, alpha, 0)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(superimposed_img)
    plt.title("Grad-CAM")
    plt.axis("off")
    plt.show()

In [ ]:
sample_folder = os.path.join(test_dir, "PNEUMONIA")
sample_img_path = os.path.join(sample_folder, os.listdir(sample_folder)[0])

img_array = get_img_array(sample_img_path, IMG_SIZE)
last_conv_layer_name = "conv5_block3_out"
heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer_name)
display_gradcam(sample_img_path, heatmap)

## SHAP

In [ ]:
background_batch, _ = next(train_data)
background = background_batch[:10]

explain_batch, _ = next(test_data)
image_to_explain = explain_batch[:1]

explainer = shap.DeepExplainer(model, background)
shap_values = explainer.shap_values(image_to_explain)

if isinstance(shap_values, list):
    shap_to_plot = shap_values[0]
else:
    shap_to_plot = shap_values

display_img = image_to_explain.copy()
display_img = (display_img - display_img.min()) / (display_img.max() - display_img.min() + 1e-8)

shap.image_plot([shap_to_plot], display_img)